[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/langchain-ai/langchain-academy/blob/main/module-1/deployment.ipynb) [![Open in LangChain Academy](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/66e9eba12c7b7688aa3dbb5e_LCA-badge-green.svg)](https://academy.langchain.com/courses/take/intro-to-langgraph/lessons/58239303-lesson-8-deployment)

# 部署

## 回顾

我们构建了一个具有记忆功能的 agent：

* `行动` - 让模型调用特定的工具
* `观察` - 将工具输出传回模型
* `推理` - 让模型对工具输出进行推理，以决定下一步要做什么（例如，调用另一个工具或直接响应）
* `持久化状态` - 使用内存检查点机制来支持有中断的长时间对话

## 目标

现在，我们将介绍如何将我们的 agent 实际部署到本地 Studio 和 `LangGraph Cloud`。

In [ ]:
%%capture --no-stderr
%pip install --quiet -U langgraph_sdk langchain_core

## 概念

有几个核心概念需要理解 -

`LangGraph` —
- Python 和 JavaScript 库
- 允许创建 agent 工作流

`LangGraph API` —
- 封装图代码
- 提供任务队列用于管理异步操作
- 提供持久化功能以维护跨交互的状态

`LangGraph Cloud` --
- LangGraph API 的托管服务
- 允许从 GitHub 仓库部署图
- 还为已部署的图提供监控和追踪功能
- 通过每个部署的唯一 URL 访问

`LangGraph Studio` --
- LangGraph 应用程序的集成开发环境（IDE）
- 使用 API 作为后端，允许实时测试和探索图
- 可以在本地运行或与云部署一起使用

`LangGraph SDK` --
- 用于以编程方式与 LangGraph 图交互的 Python 库
- 为使用图提供一致的接口，无论是在本地还是云端服务
- 允许创建客户端、访问助手、线程管理和执行运行

## 本地测试

**⚠️ 免责声明**

自这些视频拍摄以来，我们已经更新了 Studio，使其可以在本地运行并在浏览器中打开。这现在是运行 Studio 的首选方式（而不是像视频中显示的使用桌面应用程序）。请参阅 [此处](https://langchain-ai.github.io/langgraph/concepts/langgraph_studio/#local-development-server) 关于本地开发服务器的文档和 [此处](https://langchain-ai.github.io/langgraph/how-tos/local-studio/#run-the-development-server) 的更多信息。要启动本地开发服务器，请在此模块的 `/studio` 目录中的终端中运行以下命令：

```
langgraph dev
```

您应该看到以下输出：
```
- 🚀 API: http://127.0.0.1:2024
- 🎨 Studio UI: https://smith.langchain.com/studio/?baseUrl=http://127.0.0.1:2024
- 📚 API Docs: http://127.0.0.1:2024/docs
```

打开浏览器并导航到 Studio UI：`https://smith.langchain.com/studio/?baseUrl=http://127.0.0.1:2024`。

In [ ]:
if 'google.colab' in str(get_ipython()):
    raise Exception("抱歉，LangGraph Studio 目前不支持 Google Colab")

In [ ]:
from langgraph_sdk import get_client

In [ ]:
# 这是本地开发服务器的 URL
URL = "http://127.0.0.1:2024"
client = get_client(url=URL)

# 搜索所有托管的图
assistants = await client.assistants.search()

In [ ]:
assistants[-3]

In [ ]:
# 我们创建一个线程来跟踪运行状态
thread = await client.threads.create()

现在，我们可以使用 [client.runs.stream](https://langchain-ai.github.io/langgraph/concepts/low_level/#stream-and-astream) 运行我们的 agent，参数包括：

* `thread_id`
* `graph_id`
* `input`
* `stream_mode`

我们将在未来的模块中深入讨论流式传输。

现在，只需要认识到我们正在使用 `stream_mode="values"` [流式传输](https://langchain-ai.github.io/langgraph/cloud/how-tos/stream_values/) 图的每一步后状态的完整值。

状态被捕获在 `chunk.data` 中。

In [ ]:
from langchain_core.messages import HumanMessage

# 输入
input = {"messages": [HumanMessage(content="将 3 乘以 2。")]}

# 流式传输
async for chunk in client.runs.stream(
        thread['thread_id'],
        "agent",
        input=input,
        stream_mode="values",
    ):
    if chunk.data and chunk.event != "metadata":
        print(chunk.data['messages'][-1])

## 云端测试

我们可以通过 LangSmith 部署到云端，如 [此处](https://langchain-ai.github.io/langgraph/cloud/quick_start/#deploy-from-github-with-langgraph-cloud) 所述。

### 在 GitHub 上创建新仓库

* 转到您的 GitHub 账户
* 点击右上角的 "+" 图标并选择 `"New repository"`
* 为您的仓库命名（例如，`langchain-academy`）

### 将您的 GitHub 仓库添加为远程仓库

* 回到您在本课程开始时克隆 `langchain-academy` 的终端
* 将您新创建的 GitHub 仓库添加为远程仓库

```
git remote add origin https://github.com/your-username/your-repo-name.git
```
* 推送到它
```
git push -u origin main
```

### 将 LangSmith 连接到您的 GitHub 仓库

* 转到 [LangSmith](hhttps://smith.langchain.com/)
* 点击左侧 LangSmith 面板上的 `deployments` 选项卡
* 添加 `+ New Deployment`
* 然后，选择您刚为课程创建的 Github 仓库（例如，`langchain-academy`）
* 将 `LangGraph API config file` 指向其中一个 `studio` 目录
* 例如，对于 module-1 使用：`module-1/studio/langgraph.json`
* 设置您的 API 密钥（例如，您可以直接从您的 `module-1/studio/.env` 文件复制）

![Screenshot 2024-09-03 at 11.35.12 AM.png](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/66dbad4fd61c93d48e5d0f47_deployment2.png)

### 使用您的部署

然后我们可以通过几种不同的方式与我们的部署交互：

* 使用 [SDK](https://langchain-ai.github.io/langgraph/cloud/quick_start/#use-with-the-sdk)，如之前一样。
* 使用 [LangGraph Studio](https://langchain-ai.github.io/langgraph/cloud/quick_start/#interact-with-your-deployment-via-langgraph-studio)。

![Screenshot 2024-08-23 at 10.59.36 AM.png](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/66dbad4fa159a09a51d601de_deployment3.png)

要在此笔记本中使用 SDK，只需确保设置了 `LANGSMITH_API_KEY`！

In [ ]:
import os, getpass

def _set_env(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ")

_set_env("LANGSMITH_API_KEY")

In [ ]:
# 将此替换为您部署的图的 URL
URL = "https://langchain-academy-8011c561878d50b1883f7ed11b32d720.default.us.langgraph.app"
client = get_client(url=URL)

# 搜索所有托管的图
assistants = await client.assistants.search()

In [ ]:
# 选择 agent
agent = assistants[0]

In [ ]:
agent

In [ ]:
from langchain_core.messages import HumanMessage

# 我们创建一个线程来跟踪运行状态
thread = await client.threads.create()

# 输入
input = {"messages": [HumanMessage(content="将 3 乘以 2。")]}

# 流式传输
async for chunk in client.runs.stream(
        thread['thread_id'],
        "agent",
        input=input,
        stream_mode="values",
    ):
    if chunk.data and chunk.event != "metadata":
        print(chunk.data['messages'][-1])